# COLMAP Sparse + Dense Reconstruction

In [ ]:
!rm -rf /content/colmap_ws/

In [ ]:
!nvidia-smi


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# --- EDIT THESE ---
DATASET_ROOT = "/content/drive/MyDrive/images/images_set2"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/images/colmap_output2"

WORK_DIR = "/content/colmap_ws"
IMAGES_DIR = os.path.join(WORK_DIR, "images")
DB_PATH = os.path.join(WORK_DIR, "database.db")
SPARSE_DIR = os.path.join(WORK_DIR, "sparse")
DENSE_DIR = os.path.join(WORK_DIR, "dense")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(SPARSE_DIR, exist_ok=True)
os.makedirs(DENSE_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print("Dataset:", DATASET_ROOT)
print("Workspace:", WORK_DIR)


## 2. Install COLMAP (prebuilt, conda-forge — CUDA enabled)


In [ ]:
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /content/miniforge.sh
!bash /content/miniforge.sh -b -p /content/miniforge3
!/content/miniforge3/bin/mamba install -y -c conda-forge colmap


In [ ]:
!/content/miniforge3/bin/mamba install -y -c conda-forge openimageio

In [ ]:
import os

os.environ["PATH"] = "/content/miniforge3/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = "/content/miniforge3/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

!colmap -h | head -5


## 3. Flatten dataset into a single COLMAP images folder



In [ ]:
import shutil

frame_dirs = sorted(
    d for d in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, d))
)
print(f"Found {len(frame_dirs)} frame folders")

rgb1_names, rgb2_names = [], []

for tag, prefix, out_list in (
    ("rgb1.png", "rgb1_", rgb1_names),
    ("rgb2.png", "rgb2_", rgb2_names),
):
    for frame in frame_dirs:
        src_dir = os.path.join(DATASET_ROOT, frame)
        src = os.path.join(src_dir, tag)
        if not os.path.exists(src):
            print(f"  WARNING: missing {src}")
            continue
        dst_name = f"{prefix}{frame}.png"
        dst = os.path.join(IMAGES_DIR, dst_name)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        out_list.append(dst_name)

print(f"rgb1 images: {len(rgb1_names)}")
print(f"rgb2 images: {len(rgb2_names)}")
assert len(rgb1_names) == len(rgb2_names) == len(frame_dirs), \
    "Mismatch between frame folders and copied images -- check WARNINGs above."

rgb1_list_path = os.path.join(WORK_DIR, "rgb1_list.txt")
rgb2_list_path = os.path.join(WORK_DIR, "rgb2_list.txt")
with open(rgb1_list_path, "w") as f:
    f.write("\n".join(rgb1_names))
with open(rgb2_list_path, "w") as f:
    f.write("\n".join(rgb2_names))

print(open(rgb1_list_path).read())
print("---")
print(open(rgb2_list_path).read())


## 4. Feature extraction

In [ ]:
CAMERA_MODEL = "OPENCV"

RGB1_CAMERA_PARAMS = "595.607,594.714,941.101,458.698,-0.00257185,-0.0145025,0,0"
RGB2_CAMERA_PARAMS = "598.517,597.363,970.896,424.176,0.00464308,-0.0195245,0,0"

MAX_NUM_FEATURES = 8192


In [ ]:
!colmap feature_extractor \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --image_list_path {rgb1_list_path} \
    --camera_mode 1 \
    --ImageReader.camera_model {CAMERA_MODEL} \
    --ImageReader.camera_params "{RGB1_CAMERA_PARAMS}" \
    --SiftExtraction.max_num_features {MAX_NUM_FEATURES} \
    --SiftExtraction.estimate_affine_shape 1 \
    --SiftExtraction.domain_size_pooling 1

In [ ]:
import sqlite3

def db_check(expect_images=None, expect_cameras=None, label=""):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT camera_id, name FROM images")
    images = cur.fetchall()
    cur.execute("SELECT camera_id, model, width, height, params FROM cameras")
    cameras = cur.fetchall()
    conn.close()
    print(f"--- {label} ---")
    print(f"images ({len(images)}):")
    for row in images:
        print(" ", row)
    print(f"cameras ({len(cameras)}):")
    for row in cameras:
        print(" ", row)
    if expect_images is not None:
        assert len(images) == expect_images, f"Expected {expect_images} images, got {len(images)}"
    if expect_cameras is not None:
        assert len(cameras) == expect_cameras, f"Expected {expect_cameras} cameras, got {len(cameras)}"
    return images, cameras

db_check(expect_images=len(rgb1_names), expect_cameras=1, label="after rgb1 extraction")


In [ ]:
!colmap feature_extractor \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --image_list_path {rgb2_list_path} \
    --camera_mode 1 \
    --ImageReader.camera_model {CAMERA_MODEL} \
    --ImageReader.camera_params "{RGB2_CAMERA_PARAMS}" \
    --SiftExtraction.max_num_features {MAX_NUM_FEATURES} \
    --SiftExtraction.estimate_affine_shape 1 \
    --SiftExtraction.domain_size_pooling 1

In [ ]:
images, cameras = db_check(
    expect_images=len(rgb1_names) + len(rgb2_names),
    expect_cameras=2,
    label="after rgb2 extraction",
)

# Sanity check: rgb1 images should all share one camera_id, rgb2 another
rgb1_cam_ids = {cam_id for cam_id, name in images if name in rgb1_names}
rgb2_cam_ids = {cam_id for cam_id, name in images if name in rgb2_names}
assert len(rgb1_cam_ids) == 1, f"rgb1 images span multiple cameras: {rgb1_cam_ids}"
assert len(rgb2_cam_ids) == 1, f"rgb2 images span multiple cameras: {rgb2_cam_ids}"
assert rgb1_cam_ids != rgb2_cam_ids, "rgb1 and rgb2 ended up sharing the same camera!"
print("OK: rgb1 -> camera", rgb1_cam_ids, " rgb2 -> camera", rgb2_cam_ids)


## 4b. Configure the stereo rig (known extrinsic calibration)

In [ ]:
import cv2
import numpy as np
from scipy.spatial.transform import Rotation

RGB_PAIR_RVEC_RAW = np.array([0.0171, -0.0259, 3.1381])
RGB_PAIR_TVEC_RAW = np.array([-0.1482, 0.0009, 0.0015]).reshape(3, 1)

def undo_normalize_es(R, t, angle_threshold_deg=90.0):
    rvec, _ = cv2.Rodrigues(R)
    t = np.asarray(t, dtype=np.float64).reshape(3, 1)
    if np.degrees(np.linalg.norm(rvec)) < angle_threshold_deg:
        return np.ascontiguousarray(R), np.ascontiguousarray(t)
    Rz_pi, _ = cv2.Rodrigues(np.array([0.0, 0.0, np.pi]))
    return np.ascontiguousarray(R @ Rz_pi), np.ascontiguousarray(-t)

def invert_transform(R, t):
    t = np.asarray(t, dtype=np.float64).reshape(3, 1)
    return np.ascontiguousarray(R.T), np.ascontiguousarray(-R.T @ t)

R_raw, _ = cv2.Rodrigues(RGB_PAIR_RVEC_RAW)
R_pose, t_pose = undo_normalize_es(R_raw, RGB_PAIR_TVEC_RAW)
R_rgb2_from_rig, t_rgb2_from_rig = invert_transform(R_pose, t_pose)

quat_xyzw = Rotation.from_matrix(R_rgb2_from_rig).as_quat()
quat_wxyz = [float(quat_xyzw[3]), float(quat_xyzw[0]), float(quat_xyzw[1]), float(quat_xyzw[2])]

print("cam_from_rig rotation (w,x,y,z):", quat_wxyz)
print("cam_from_rig translation (m):   ", t_rgb2_from_rig.flatten())

In [ ]:
import json

rig_config = [
    {
        "cameras": [
            {"image_prefix": "rgb1_", "ref_sensor": True},
            {
                "image_prefix": "rgb2_",
                "cam_from_rig_rotation": quat_wxyz,
                "cam_from_rig_translation": t_rgb2_from_rig.flatten().tolist(),
            },
        ]
    }
]

rig_config_path = os.path.join(WORK_DIR, "rig_config.json")
with open(rig_config_path, "w") as f:
    json.dump(rig_config, f, indent=2)

print(open(rig_config_path).read())


In [ ]:
!colmap rig_configurator \
    --database_path {DB_PATH} \
    --rig_config_path {rig_config_path}


## 5. Matching

In [ ]:
!colmap exhaustive_matcher \
    --database_path {DB_PATH} \
    --FeatureMatching.guided_matching 1 \
    --SiftMatching.max_ratio 0.85 \
    --TwoViewGeometry.min_num_inliers 10 \
    --TwoViewGeometry.min_inlier_ratio 0.15


In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
n_matches = cur.execute("SELECT COUNT(*) FROM matches").fetchone()[0]
n_geom = cur.execute("SELECT COUNT(*) FROM two_view_geometries").fetchone()[0]
conn.close()
print("matches rows:", n_matches)
print("two_view_geometries rows:", n_geom)
if n_geom == 0:
    print("WARNING: zero verified geometries — matching/verification is failing, "
          "not just being strict. Check images visually before loosening thresholds further.")


## 6. Sparse reconstruction (mapper)


In [ ]:
!colmap mapper \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --output_path {SPARSE_DIR} \
    --Mapper.init_min_tri_angle 2 \
    --Mapper.abs_pose_min_num_inliers 15 \
    --Mapper.ba_refine_focal_length 0 \
    --Mapper.ba_refine_principal_point 0 \
    --Mapper.ba_refine_extra_params 0 \
    --Mapper.ba_refine_sensor_from_rig 1


In [ ]:
models = sorted(os.listdir(SPARSE_DIR))
print("Reconstructed models:", models)
assert models, "No sparse model produced — check matcher output above."

BEST_MODEL = os.path.join(SPARSE_DIR, models[0])  # adjust if multiple models and 0 isn't largest
!colmap model_analyzer --path {BEST_MODEL}


In [ ]:
!colmap model_converter \
    --input_path {BEST_MODEL} \
    --output_path {BEST_MODEL} \
    --output_type TXT

print(open(os.path.join(BEST_MODEL, "cameras.txt")).read())

lines = open(os.path.join(BEST_MODEL, "images.txt")).read().splitlines()
image_lines = [l for l in lines if l and not l.startswith("#")][::2]
print(f"{len(image_lines)} registered images:")
for l in image_lines:
    print(" ", l.split()[-1])


In [ ]:
import numpy as np

def read_colmap_image_poses(images_txt_path):
    poses = {}
    lines = [l for l in open(images_txt_path).read().splitlines() if l and not l.startswith("#")]
    for line in lines[::2]:
        parts = line.split()
        qw, qx, qy, qz = map(float, parts[1:5])
        tx, ty, tz = map(float, parts[5:8])
        name = parts[-1]
        R = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()  # world_from_cam rotation (COLMAP stores cam_from_world)
        t = np.array([tx, ty, tz])
        cam_center = -R.T @ t  # camera center in world coordinates
        poses[name] = cam_center
    return poses

poses = read_colmap_image_poses(os.path.join(BEST_MODEL, "images.txt"))
rgb1_frames = {n[len("rgb1_"):] for n in poses if n.startswith("rgb1_")}
rgb2_frames = {n[len("rgb2_"):] for n in poses if n.startswith("rgb2_")}
common = sorted(rgb1_frames & rgb2_frames)

baselines = []
for frame in common:
    c1 = poses[f"rgb1_{frame}"]
    c2 = poses[f"rgb2_{frame}"]
    baselines.append(np.linalg.norm(c1 - c2))

## 7. Dense reconstruction — undistort

In [ ]:
DENSE_WORKSPACE = DENSE_DIR

!colmap image_undistorter \
    --image_path {IMAGES_DIR} \
    --input_path {BEST_MODEL} \
    --output_path {DENSE_WORKSPACE} \
    --output_type COLMAP


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

undistorted_dir = os.path.join(DENSE_WORKSPACE, "images")
undist_files = sorted(os.listdir(undistorted_dir))
print(f"{len(undist_files)} undistorted images:", undist_files)

sample_img_path = os.path.join(undistorted_dir, undist_files[0])
img = Image.open(sample_img_path)
print("Original capture size: (1920, 1080)")
print("Undistorted size:", img.size)

plt.figure(figsize=(16, 5))
plt.imshow(img)
plt.title(f"{undist_files[0]}  size={img.size}")
plt.axis("off")
plt.show()


## 8. Dense reconstruction — PatchMatch stereo + fusion


In [ ]:
!colmap patch_match_stereo \
    --workspace_path {DENSE_WORKSPACE} \
    --workspace_format COLMAP \
    --PatchMatchStereo.geom_consistency true


### Depth map coverage check

In [ ]:
import numpy as np

def read_colmap_depth(path):
    with open(path, "rb") as f:
        header = b""
        while header.count(b"&") < 3:
            header += f.read(1)
        w, h, c = map(int, header.decode().strip("&").split("&"))
        data = np.fromfile(f, dtype=np.float32)
        return data.reshape(h, w, c) if c > 1 else data.reshape(h, w)

depth_dir = os.path.join(DENSE_WORKSPACE, "stereo", "depth_maps")
depth_files = [f for f in os.listdir(depth_dir) if f.endswith(".geometric.bin")]
print(f"{len(depth_files)} geometric depth maps")

for fname in depth_files:
    depth = read_colmap_depth(os.path.join(depth_dir, fname))
    valid = np.sum(depth > 0)
    total = depth.size
    print(f"{fname}: shape={depth.shape}  valid={valid}/{total} ({100*valid/total:.1f}%)")


In [ ]:
fused_ply = os.path.join(DENSE_WORKSPACE, "fused.ply")

# Loosened fusion thresholds given low view count / low texture:
# fewer views required to agree, more tolerance on reprojection/depth/normal consistency.
!colmap stereo_fusion \
    --workspace_path {DENSE_WORKSPACE} \
    --workspace_format COLMAP \
    --input_type geometric \
    --output_path {fused_ply} \
    --StereoFusion.min_num_pixels 2 \
    --StereoFusion.max_reproj_error 4 \
    --StereoFusion.max_depth_error 0.1 \
    --StereoFusion.max_normal_error 15

print("Fused point cloud exists:", os.path.exists(fused_ply))


## 9. (Optional) Mesh the fused point cloud

In [ ]:
mesh_path = os.path.join(DENSE_WORKSPACE, "meshed_poisson.ply")
!colmap poisson_mesher \
    --input_path {fused_ply} \
    --output_path {mesh_path}
print("Mesh exists:", os.path.exists(mesh_path))


## 10. Copy results back to Drive


In [ ]:
def copy_tree(src, dst_name):
    dst = os.path.join(DRIVE_OUTPUT_DIR, dst_name)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print("Copied", src, "->", dst)

copy_tree(SPARSE_DIR, "sparse")

os.makedirs(os.path.join(DRIVE_OUTPUT_DIR, "dense"), exist_ok=True)
for fname in ("fused.ply", "meshed_poisson.ply"):
    src = os.path.join(DENSE_WORKSPACE, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_OUTPUT_DIR, "dense", fname))
        print("Copied", fname)


## 11. Quick visualization


In [ ]:
import sys
!{sys.executable} -m pip install -q plyfile
import plyfile
print("plyfile OK:", plyfile.__file__)


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plyfile import PlyData

def read_ply_xyz_rgb(path, max_points=200_000):
    ply = PlyData.read(path)
    v = ply["vertex"]
    xyz = np.stack([v["x"], v["y"], v["z"]], axis=1)
    if all(c in v.data.dtype.names for c in ("red", "green", "blue")):
        rgb = np.stack([v["red"], v["green"], v["blue"]], axis=1)
    else:
        rgb = np.full((xyz.shape[0], 3), 180)
    if xyz.shape[0] > max_points:
        idx = np.random.choice(xyz.shape[0], max_points, replace=False)
        xyz, rgb = xyz[idx], rgb[idx]
    return xyz, rgb

xyz, rgb = read_ply_xyz_rgb(fused_ply)
print(f"{xyz.shape[0]} points loaded")

# Drop top 1% by z as likely outliers before plotting
z_thresh = np.percentile(xyz[:, 2], 99)
mask = xyz[:, 2] < z_thresh
xyz_clean, rgb_clean = xyz[mask], rgb[mask]
print(f"{xyz_clean.shape[0]} points after outlier filtering")

colors = [f"rgb({r},{g},{b})" for r, g, b in rgb_clean]

fig = go.Figure(data=[go.Scatter3d(
    x=xyz_clean[:, 0], y=xyz_clean[:, 1], z=xyz_clean[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=colors),
)])
fig.update_layout(scene=dict(aspectmode="data"), margin=dict(l=0, r=0, t=0, b=0))
fig.show()
